# TempGuru Event Staffing Tools

This notebook walks through using the [TempGuru](https://tempguru.co) tool spec with a LlamaIndex agent.

TempGuru provides W-2 compliant event staffing across 345+ US and Canadian cities. The five data tools (coverage, roles, lead times, rates, state compliance) are **read-only and need no API key**. A sixth, opt-in tool submits a confirmed staffing plan for a human-reviewed quote.

In [ ]:
%pip install llama-index-tools-tempguru llama-index-llms-openai

## Inspect the tool spec

No configuration is required — the spec talks to TempGuru's public API.

In [ ]:
from llama_index.tools.tempguru import TempGuruToolSpec

tool_spec = TempGuruToolSpec()

for tool in tool_spec.to_tool_list():
    print(tool.metadata.name)

You can also call the data tools directly, without an agent:

In [ ]:
tool_spec.event_staffing_pricing(role="brand-ambassadors", city="Boston")

## Use the tools in an agent

In [ ]:
import os

from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

os.environ["OPENAI_API_KEY"] = "sk-..."

agent = FunctionAgent(
    tools=tool_spec.to_tool_list(),
    llm=OpenAI(model="gpt-4o"),
)

response = await agent.run(
    "I'm planning a trade show in Boston on August 12. What would 10 brand"
    " ambassadors cost per hour, and how far ahead should I book?"
)
print(response)

## Read-only mode

`submit_event_staffing_quote_request` sends contact and event details to TempGuru's CRM so a coordinator can respond with a quote. It creates no reservation and requires no payment, but it *is* a write — if you want a strictly read-only tool set, drop it with one flag:

In [ ]:
read_only_spec = TempGuruToolSpec(include_quote_submission=False)

for tool in read_only_spec.to_tool_list():
    print(tool.metadata.name)

## Notes

- Rate ranges are planning estimates, never binding quotes.
- Availability responses are lead-time guidance, never reservations.
- Compliance summaries are operational guidance, not legal advice.
- API reference: [mcp.tempguru.co/openapi.json](https://mcp.tempguru.co/openapi.json) · Docs: [tempguru.co/ai](https://tempguru.co/ai)